# Regression examples

In [1]:
from sklearn.datasets import load_diabetes
import itertools
import sys

sys.path.append("..")

from ai_toolkit import (
    BaseDataset, 
    ConfigFactory,
    RidgeRegressionModel,
    KNNRegressorModel,
    LightGBMRegressorModel, 
    get_all_regression_models, 
    RegressionModelTrainer, 
    lazypredict_regression,
    EnsembleVotingRegressorModel,
    EnsembleStackingRegressorModel,
)

## Example data

In [2]:
class RegDataset(BaseDataset):
    """Dataset for diabetes regression task.
    https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html
    """
    def __init__(self):
        """Initialize the clf dataset."""

        super().__init__()

    def load_data(self):
        """Load the diabetes dataset for regression task."""
        
        self.X, self.y = load_diabetes(return_X_y=True, as_frame=True)
        self.X_test = self.X.head()

In [3]:
RDataset = RegDataset()
RDataset.load_data()
RDataset.preprocess()
X_reg, y_reg, X_test_reg = RDataset.get_data()

## Configuration

In [ ]:
config_factory = ConfigFactory()
config_factory.training.experiment_name = "ai_toolkit_regression_experiment"
config_factory.training.optimize_metric = "root_mean_squared_error"
configs = config_factory.get_config()

# config_factory.save_config_file("config_development.yaml")
config = configs.training

## Training and evaluation

### Train one example model

In [ ]:
base_model = RidgeRegressionModel()

# Create a regression model trainer
trainer = RegressionModelTrainer(
    base_model=base_model,
    config_factory=config_factory,
)

In [ ]:
# Train and optimize the model
best_model, mean_metrics = trainer.train_and_optimize(
    X=X_reg, 
    y=y_reg, 
)

# y_pred = trainer.predict(X_test)

### Train all regression models

In [ ]:
base_models = get_all_regression_models()

for base_model in base_models.values():

    # Create a regression model trainer
    trainer = RegressionModelTrainer(
        base_model=base_model,
        config_factory=config_factory,
    )

    # Train and optimize the model
    best_model, mean_metrics = trainer.train_and_optimize(
        X=X_reg, 
        y=y_reg, 
    )

In [ ]:
df_mean_results = lazypredict_regression(
    X=X_reg, 
    y=y_reg, 
    n_splits=config.n_splits,
    random_state=config.random_state,
)

# print(df_mean_results.to_string())
df_mean_results

### Ensemble

In [ ]:
# (Model, mlflow run_id) pairs
model_pool = [
    (RidgeRegressionModel(), "8176863ad9b344c6be3f00f9c5987737"),
    (KNNRegressorModel(), "41444ec16eba4a60b46aff3ad933e178"),
    (LightGBMRegressorModel(), "db8511fbb32b4582be011d3efb1daec4"),
]

meta_model = RidgeRegressionModel()

In [ ]:
combinations = []
 
for model in range(2, len(model_pool) + 1):
    combinations.extend(itertools.combinations(model_pool, model))

#### Voting | Train all combinations

In [ ]:
for combination in combinations:
    models = list(combination)

    base_model = EnsembleVotingRegressorModel(
        models=models,
    )

    # Create a regression model trainer
    trainer = RegressionModelTrainer(
        base_model=base_model,
        config_factory=config_factory,
    )
    
    # Train and optimize the model
    best_model, mean_metrics = trainer.train_and_optimize(
        X=X_reg, 
        y=y_reg, 
    )

    # y_pred = trainer.predict(X_test)

#### Stacking | Train all combinations

In [ ]:
for combination in combinations:
    models = list(combination)

    base_model = EnsembleStackingRegressorModel(
        models=models,
        meta_model=meta_model,
    )

    # Create a regression model trainer
    trainer = RegressionModelTrainer(
        base_model=base_model,
        config_factory=config_factory,
    )

    # Train and optimize the model
    best_model, mean_metrics = trainer.train_and_optimize(
        X=X_reg, 
        y=y_reg, 
    )

    # y_pred = trainer.predict(X_test)